In [ ]:
import numpy as np
import numpy.random
import math
import random
from enum import Enum
from enum import IntEnum
from enum import auto

TRAINING_BOND_GAIN = 7

# URA: training level goes up for every 4 trainings of that type (training_count[room] / 4)
ROOM_URA = {
    "speed"   : [np.array([11, 0, 6, 0, 0, 4]),
                 np.array([12, 0, 6, 0, 0, 4]),
                 np.array([13, 0, 6, 0, 0, 4]),
                 np.array([14, 0, 7, 0, 0, 4]),
                 np.array([15, 0, 8, 0, 0, 4]),],
    "stamina" : [np.array([0, 11, 0, 6, 0, 4]),
                 np.array([0, 12, 0, 6, 0, 4]),
                 np.array([0, 13, 0, 6, 0, 4]),
                 np.array([0, 14, 0, 7, 0, 4]),
                 np.array([0, 15, 0, 8, 0, 4]),],
    "power"   : [np.array([0, 6, 9, 0, 0, 4]),
                 np.array([0, 6, 10, 0, 0, 4]),
                 np.array([0, 6, 11, 0, 0, 4]),
                 np.array([0, 7, 12, 0, 0, 4]),
                 np.array([0, 8, 13, 0, 0, 4]),],
    "guts"    : [np.array([5, 0, 5, 8, 0, 4]),
                 np.array([5, 0, 5, 9, 0, 4]),
                 np.array([5, 0, 5, 10, 0, 4]),
                 np.array([6, 0, 5, 11, 0, 4]),
                 np.array([7, 0, 6, 12, 0, 4]),],
    "wisdom"  : [np.array([2, 0, 0, 0, 10, 5]),
                 np.array([2, 0, 0, 0, 11, 5]),
                 np.array([2, 0, 0, 0, 12, 5]),
                 np.array([3, 0, 0, 0, 13, 5]),
                 np.array([4, 0, 0, 0, 14, 5]),],
    "wisdom"  : [np.array([2, 0, 0, 0, 10, 5]),
                 np.array([2, 0, 0, 0, 11, 5]),
                 np.array([2, 0, 0, 0, 12, 5]),
                 np.array([3, 0, 0, 0, 13, 5]),
                 np.array([4, 0, 0, 0, 14, 5]),],
    "mia"  :     np.zeros((5, 6)),
}

ROOM_INDEX_TO_ROOM_KEY = ["speed", "stamina", "power", "guts", "wisdom"]
STAT_INDEX_TO_STAT_KEY = ["speed", "stamina", "power", "guts", "wisdom", "sp"]

# Returns an int representing the randomly selected training room
# Possible return values: 0 (Speed), 1 (Stamina), 2 (Power), 3 (Guts), 4 (Wisdom), 5 (MIA)
def getRandomRoom(room_type, specialty_rate):
    r = np.random.randint(550 + specialty_rate)
    room_weights = [100, 200, 300, 400, 500, 550]

    for i in range(0, len(room_weights)):
        if i >= room_type:
            room_weights[i] += specialty_rate

        if r <= room_weights[i]:
            return i

# Returns a list containing the flat bonuses of a support card
def getSupportFlatBonusAsList(support):
    return [support.get("speed_bonus", 0),
            support.get("stamina_bonus", 0),
            support.get("power_bonus", 0),
            support.get("guts_bonus", 0),
            support.get("wisdom_bonus", 0),
            support.get("sp_bonus", 0)]

# Returns a list containing the initial bonuses of a support card
def getSupportInitialBonusAsList(support):
    return [support.get("initial_speed", 0),
            support.get("initial_stamina", 0),
            support.get("initial_power", 0),
            support.get("initial_guts", 0),
            support.get("initial_wisdom", 0),
            0]

# Break ties in the order of: highest stat, highest sp, random between the tied
# Returns an integer representing the index of the final room
def breakRoomTies(totals, rooms):
    max_room_indices = [i for i, x in enumerate(totals) if x == max(totals)]

    # First tie breaker for highest stat
    if len(max_room_indices) == 1:
        return max_room_indices[0]
    else:
        # If there are multiple rooms with the max stat values, second tie breaker for highest sp
        total_SPs = [rooms[total_index][5] for total_index in max_room_indices]
        max_SP_indices = [i for i, x in enumerate(total_SPs) if x == max(total_SPs)]

        if len(max_SP_indices) == 1:
            return max_room_indices[0]
        # If there are multiple rooms with the same max stat and sp values, randomly pick one
        else:
            return random.choice(max_SP_indices)

def getRoomWithHighestStat(rooms):
    stat_totals = [(sum(room) - room[5]) for room in rooms]
    return breakRoomTies(stat_totals, rooms)

def getRoomWithHighestTotal(rooms):
    totals = [(sum(room)) for room in rooms]
    return breakRoomTies(totals, rooms)

# Takes a dictionary of supports and applies any starting effects, as well as any unique starting effects (e.g. Oguri Cap Int SSR)
def ApplySupportInitialEffects(supports, stats):
    for support_key, support in supports.items():
        stats = np.add(stats, getSupportInitialBonusAsList(support))

        unique_effect = support.get("unique_effect", 0)
        match unique_effect:
            case "oguri_cap_int_ssr":
                for i, j in supports.items():
                    j["current_bond"] += 5

# Takes a dictionary of supports as input and checks each individual unique effect
def UpdateSupportUniqueEffects(supports):
    for support_key, support in supports.items():
        unique_effect = support.get("unique_effect", "")

        # Jungle Pocket Speed SSR
        match unique_effect:
            case "jungle_pocket_speed_ssr":
                if support.get("current_bond") >= 80:
                    support["speed_bonus"] = 3
            case "symboli_rudolf_guts_ssr":
                if support.get("current_bond") >= 80:
                    support["speed_bonus"] = 2


def train(n, context):
    turn = 0
    training_count = np.array([0, 0, 0, 0, 0])
    stats = np.array([0, 0, 0, 0, 0, 0])

    scenario_rooms = []
    match context.get("scenario"):
        case Scenario.URA:
            scenario_rooms = ROOM_URA
        case _:
            scenario_rooms = ROOM_URA

    # Contain the state (bonds, unique effects) of the supports
    supports = {}
    for support_key in context.get("deck"):
        supports[support_key] = support_list.get(support_key)
        supports[support_key]["current_bond"] = supports[support_key].get("initial_bond")

    # Add any initial stat bonuses
    ApplySupportInitialEffects(supports, stats)

    # Begin the training loop
    while turn < n:
        turn = turn + 1
        rooms_with_support_keys = [[] for i in range(5)]

        # Distribute supports into training rooms
        for i, (support_key, support) in enumerate(supports.items()):
            roomIndex = getRandomRoom(int(support.get("card_type")), support.get("specialty_rate"))
            if roomIndex < 5:
                rooms_with_support_keys[roomIndex].append(support_key)

        # Calculate the total stat values for each training
        final_rooms = []
        for room_index, room in enumerate(rooms_with_support_keys):
            total_friendship_bonus = 1
            total_training_bonus = 1
            total_motivation_bonus = 0
            total_flat_bonuses = np.zeros(6)
            total_bonuses = []
            room_level = min(training_count[room_index] // 4, 4)
            current_room = scenario_rooms.get(ROOM_INDEX_TO_ROOM_KEY[room_index])[room_level]

            # For each support in the current room, add their training bonuses
            for support_key in room:
                support = supports.get(support_key)

                # If the support matches the room and has 80 or more bond, add their rainbow training
                if support.get("card_type") < 5 and room_index == int(support.get("card_type")) and support.get("current_bond") >= 80:
                    total_friendship_bonus *= (1 + support.get("friendship_bonus") / 100)

                total_training_bonus += (support.get("training_bonus", 0) / 100)
                total_motivation_bonus += support.get("motivation_bonus", 0)
                flat_bonuses = [support.get(key, 0) for key in ["speed_bonus", "stamina_bonus", "power_bonus", "guts_bonus", "wisdom_bonus", "sp_bonus"]]
                total_flat_bonuses = np.add(total_flat_bonuses, flat_bonuses)

            # Remove flat bonuses if the room does not have it
            for bonus_index, flat_bonus in enumerate(total_flat_bonuses):
                if current_room[bonus_index] == 0:
                    flat_bonus = 0

                # Calculate the total stat bonuses of the room
                # (BaseValue + Sum of StatBonus) * (1 + MotivationMultiplier * (Sum of MotivationBonus)) *
                #   (Sum of TrainingBonus) * (Product of FriendshipBonus) * (1 + 0.05 * NumberOfSupportCards)
                total_bonus = math.floor((current_room[bonus_index] + flat_bonus) * (1 + .2 * total_motivation_bonus / 100) * total_training_bonus * total_friendship_bonus * (1 + .05 * len(room)))
                total_bonuses.append(total_bonus)

            final_rooms.append(total_bonuses)

        # Pick a room or strategy
        selected_room_index = -1

        match context.get("strategy"):
            case Strategy.SPEED:
                selected_room_index = 0
            case Strategy.STAMINA:
                selected_room_index = 1
            case Strategy.POWER:
                selected_room_index = 2
            case Strategy.GUTS:
                selected_room_index = 3
            case Strategy.WISDOM:
                selected_room_index = 4
            case Strategy.TURN_BY_TURN:
                # Print the game state (turn, room stats, supports)
                print("Turn " + str(turn))
                for room_index, room in enumerate(final_rooms):
                    text_array = [" - ", ROOM_INDEX_TO_ROOM_KEY[room_index],": "]
                    bonus_total = sum(room)

                    for bonus_index, bonus in enumerate(room):
                        if bonus != 0:
                            text_array.append(STAT_INDEX_TO_STAT_KEY[bonus_index] + " "  + str(bonus))
                            text_array.append(", ")

                    text_array.append("stats: " + str(bonus_total - room[5]))
                    text_array.append(" (" + ", ".join(rooms_with_support_keys[room_index]) + ")")
                    room_text = "".join(text_array)
                    print(room_text)

                # Get input for the selected room
                while True:
                    print("Pick a room. 0 = speed, 1 = stamina, 2 = power, 3 = guts, 4 = wisdom")
                    selected_room_index = input()
                    if not selected_room_index.isdigit() and selected_room_index not in [0, 1, 2, 3, 4]:
                        print("Input must be an integer from 0 to 4")
                    else:
                        selected_room_index = int(selected_room_index)
                        print()
                        break
            case Strategy.HIGHEST_STAT:
                selected_room_index = getRoomWithHighestStat(final_rooms)
            case Strategy.HIGHEST_TOTAL:
                selected_room_index = getRoomWithHighestTotal(final_rooms)
            case _:
                selected_room_index = 0

        # Increase bonds of the supports in the selected room
        for support_key in rooms_with_support_keys[selected_room_index]:
            supports[support_key]["current_bond"] = max(supports[support_key]["current_bond"] + TRAINING_BOND_GAIN, 100)

        # Add the room's final bonuses to stats
        stats = np.add(stats, final_rooms[selected_room_index])
        training_count[selected_room_index] += 1

        # Check the supports dictionary and apply any unique effects
        UpdateSupportUniqueEffects(supports)

    return stats

In [ ]:
class CardType(IntEnum):
    SPEED = 0
    STAMINA = 1
    POWER = 2
    GUTS = 3
    WISDOM = 4
    FRIEND = 5
    GROUP = 6

class Scenario(Enum):
    URA = auto()
    AOHARU = auto()
    MANT = auto()
    GL = auto()
    GM = auto()

class Strategy(Enum):
    SPEED = auto()
    STAMINA = auto()
    POWER = auto()
    GUTS = auto()
    WISDOM = auto()
    HIGHEST_STAT = auto()
    HIGHEST_TOTAL = auto()
    TURN_BY_TURN = auto()
    BOND_THEN_HIGHEST_STAT = auto()
    BOND_THEN_HIGHEST_TOTAL = auto()

# unique motivation and training bonus are additive, but unique friendship is multiplicative
support_list = {
    "generic_speed": {"card_type":CardType.SPEED, "initial_bond":0, "specialty_rate":20, "friendship_bonus":20, "speed_bonus":1},
    "generic_stamina": {"card_type":CardType.STAMINA, "initial_bond":0, "specialty_rate":20, "friendship_bonus":20, "stamina_bonus":1},
    "generic_power": {"card_type":CardType.POWER, "initial_bond":0, "specialty_rate":20, "friendship_bonus":20, "power_bonus":1},
    "generic_guts": {"card_type":CardType.GUTS, "initial_bond":0, "specialty_rate":20, "friendship_bonus":20, "guts_bonus":1},
    "generic_wisdom": {"card_type":CardType.WISDOM, "initial_bond":0, "specialty_rate":20, "friendship_bonus":20, "wisdom_bonus":1},
    "generic_friend": {"card_type":CardType.FRIEND, "initial_bond":0, "friendship_bonus":20},
    "generic_group": {"card_type":CardType.GROUP, "initial_bond":0, "friendship_bonus":20},

    "kitasan_black_speed_ssr": {"card_type":CardType.SPEED, "initial_bond":35, "specialty_rate":116, "friendship_bonus":25, "training_bonus":15, "motivation_bonus":30, "race_bonus":5, "power_bonus":1, "unique_effect": "kitasan_black_speed_ssr"},
    "jungle_pocket_speed_ssr": {"card_type":CardType.SPEED, "initial_bond":30, "specialty_rate":65, "friendship_bonus":30, "training_bonus":10, "race_bonus":10, "power_bonus":1, "sp_bonus":1, "initial_power": 35, "unique_effect": "jungle_pocket_speed_ssr"},

    "super_creek_stamina_ssr": {"card_type":CardType.STAMINA, "initial_bond":30, "specialty_rate":62, "friendship_bonus":37.5, "training_bonus":15, "race_bonus":10, "stamina_bonus":1, "initial_stamina":35},

    "haru_urara_guts_ssr": {"card_type":CardType.GUTS, "initial_bond":0, "specialty_rate":50, "friendship_bonus":32, "training_bonus":15, "motivation_bonus":30, "race_bonus":10, "sp_bonus":1, "initial_guts": 35},
    "symboli_rudolf_guts_ssr": {"card_type":CardType.GUTS, "initial_bond":25, "specialty_rate":65, "friendship_bonus":20, "training_bonus":10, "race_bonus":5, "power_bonus":1, "guts_bonus":2, "initial_speed":20},

    "mejiro_ramonu_wisdom_ssr": {"card_type":CardType.WISDOM, "initial_bond":20, "specialty_rate":50, "friendship_bonus":35, "training_bonus":20, "race_bonus":5, "wisdom_bonus":2, "sp_bonus":1, "wisdom_recovery": 5},
    "fine_motion_wisdom_ssr": {"card_type":CardType.WISDOM, "initial_bond":15, "specialty_rate":35, "friendship_bonus":37.5, "training_bonus":15, "motivation_bonus":30, "race_bonus":10, "wisdom_bonus":1, "wisdom_recovery":5, "initial_wisdom":35},

}

In [ ]:
generic_deck = ["generic_speed", "generic_stamina", "generic_power", "generic_guts", "generic_wisdom"]
strong_deck = ["kitasan_black_speed_ssr", "jungle_pocket_speed_ssr", "haru_urara_guts_ssr", "symboli_rudolf_guts_ssr", "mejiro_ramonu_wisdom_ssr", "fine_motion_wisdom_ssr"]

sample = {
    "deck": strong_deck,
    "number_objective_races": 10, # Unused
    "initial_bond_bonus": 0,
    "scenario":  Scenario.URA,
    "strategy": Strategy.HIGHEST_STAT,
}

sample_test = train(50, sample)
for i, stat in enumerate(sample_test):
    print(STAT_INDEX_TO_STAT_KEY[i] + ": " + str(stat))
print("total stats: " + str(sum(sample_test) - sample_test[5]))


speed: 875
stamina: 88
power: 567
guts: 244
wisdom: 247
sp: 510
total stats: 2021
